# GenerativeAURORA — Step 3: Build Daily Sequences

This notebook is the Google Colab/Jupyter version of `step3_build_sequences.py`.

## Purpose
Convert the cleaned daytime NSRDB dataset into fixed-length daily sequences for the GenerativeAURORA diffusion model.

The approach is:

- **One sequence per day per site**
- **Condition (`X`)**: daytime solar and meteorological variables
- **Target (`Y`)**: the day's daytime GHI trajectory
- **Regime (`R`)**: dominant sky regime of the day
- **Site (`S`)**: encoded study-site identifier

Daily sequences are padded or truncated so every sample has the same dimensions.

## Workflow
```text
download_nsrdb
      ↓
merge_nsrdb
      ↓
eda_nsrdb
      ↓
step3_build_sequences  ← THIS NOTEBOOK
      ↓
step4_build_model
      ↓
step5_train
```

**Input:** `nsrdb_merged.parquet`

**Outputs:** `sequences_train.npz`, `sequences_val.npz`, `sequences_test.npz`, `scaler_params.json`, and `sequence_summary.json`.


## 1. Install and import required packages


In [ ]:
!pip -q install pandas numpy pyarrow

import os
import json
import numpy as np
import pandas as pd

print("Packages imported successfully.")


Packages imported successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configure input/output paths

The original Python script uses a Windows-specific output path. In Colab, we use the relative `Dataset/` directory instead.


In [ ]:
PARQUET_FILE = "/content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet"
OUTPUT_DIR = "Dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input file :", PARQUET_FILE)
print("Output dir :", OUTPUT_DIR)


Input file : /content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet
Output dir : Dataset


## 3. Define sequence parameters and model inputs

### Fixed sequence length
- `LOOKBACK = 16` hours
- `HORIZON = 16` hours

Shorter daily sequences are zero-padded; longer sequences are truncated.

### Conditioning features
The original implementation uses ten conditioning variables:

`GHI`, `DHI`, `DNI`, `WIND SPEED`, `TEMPERATURE`, `RELATIVE HUMIDITY`, `PRESSURE`, `CLEARSKY GHI`, `SOLAR ZENITH ANGLE`, and `CLEARNESS_INDEX`.

### Target
`GHI`

### Categorical encodings
Clear → 0, Cloudy → 1, Overcast → 2.

The five sites are encoded from 0 to 4.


In [ ]:
LOOKBACK = 16
HORIZON = 16

CONDITION_FEATURES = [
    "GHI",
    "DHI",
    "DNI",
    "WIND SPEED",
    "TEMPERATURE",
    "RELATIVE HUMIDITY",
    "PRESSURE",
    "CLEARSKY GHI",
    "SOLAR ZENITH ANGLE",
    "CLEARNESS_INDEX",
]

TARGET_FEATURE = "GHI"

REGIME_MAP = {
    "Clear": 0,
    "Cloudy": 1,
    "Overcast": 2,
}

SITE_MAP = {
    "Phoenix_AZ": 0,
    "LosAngeles_CA": 1,
    "Denver_CO": 2,
    "Miami_FL": 3,
    "Seattle_WA": 4,
}

TRAIN_YEARS = [2018, 2019, 2020, 2021]

print("Lookback :", LOOKBACK)
print("Horizon  :", HORIZON)
print("Condition features:", len(CONDITION_FEATURES))
print("Training years:", TRAIN_YEARS)
print("Validation: 2022 Jan-Jun")
print("Test: 2022 Jul-Dec")


Lookback : 16
Horizon  : 16
Condition features: 10
Training years: [2018, 2019, 2020, 2021]
Validation: 2022 Jan-Jun
Test: 2022 Jul-Dec


## 4. Load and harmonize the merged dataset

There is an important naming detail between the earlier preprocessing scripts and this script. The merged dataset may contain:

- `SITE` instead of `SITE_NAME`
- `AIR TEMPERATURE` instead of `TEMPERATURE`
- `SURFACE PRESSURE` instead of `PRESSURE`

The notebook makes these mappings explicit so the sequence builder receives the names it expects.


In [ ]:
def load_data(path):
    """
    Load the merged NSRDB dataset and harmonize required column names.
    """
    print(f"Loading {path}...")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} was not found. Run merge_nsrdb first."
        )

    df = pd.read_parquet(path)

    rename_map = {}

    if "SITE" in df.columns and "SITE_NAME" not in df.columns:
        rename_map["SITE"] = "SITE_NAME"

    if "AIR TEMPERATURE" in df.columns and "TEMPERATURE" not in df.columns:
        rename_map["AIR TEMPERATURE"] = "TEMPERATURE"

    if "SURFACE PRESSURE" in df.columns and "PRESSURE" not in df.columns:
        rename_map["SURFACE PRESSURE"] = "PRESSURE"

    if rename_map:
        df = df.rename(columns=rename_map)
        print("Column harmonization:")
        for old, new in rename_map.items():
            print(f"  {old} -> {new}")
    else:
        print("No column-name harmonization required.")

    print(
        f"Loaded {len(df):,} rows, "
        f"{len(df.columns)} columns."
    )
    print(f"Columns: {list(df.columns)}")
    print("")

    return df


df = load_data(PARQUET_FILE)


Loading /content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet...
Column harmonization:
  SITE -> SITE_NAME
Loaded 110,502 rows, 22 columns.
Columns: ['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE', 'GHI', 'DHI', 'DNI', 'WIND SPEED', 'TEMPERATURE', 'RELATIVE HUMIDITY', 'PRESSURE', 'CLOUD TYPE', 'SOLAR ZENITH ANGLE', 'CLEARSKY GHI', 'SITE_NAME', 'CLIMATE', 'LAT', 'LON', 'CLEARNESS_INDEX', 'SKY_REGIME', 'DATETIME']



## 5. Validate required columns

Before creating sequences, verify that all variables needed for conditioning, targets, dates, sites, and sky regimes are present.

The notebook stops here if required columns are missing instead of producing incomplete sequence files.


In [ ]:
required_columns = list(dict.fromkeys(
    CONDITION_FEATURES
    + [
        TARGET_FEATURE,
        "YEAR",
        "MONTH",
        "DAY",
        "HOUR",
        "DATETIME",
        "SITE_NAME",
        "SKY_REGIME",
    ]
))

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("=== Dataset validation ===")

if missing_columns:
    print("Missing required columns:")
    for col in missing_columns:
        print(f"  - {col}")

    raise ValueError(
        "Sequence construction cannot continue because required columns are missing."
    )

print("All required columns are present.")
print(f"Dataset shape: {df.shape}")


=== Dataset validation ===
All required columns are present.
Dataset shape: (110502, 22)


## 6. Fit min-max scalers using training years only

Scaling parameters are fitted using **2018–2021 only**, matching the original script.

For each feature:

\[
x_{scaled} = (x - x_{min}) / (x_{max} - x_{min})
\]

This prevents validation/test data from being used when fitting the scaler. The fitted parameters are saved for reuse.


In [ ]:
def compute_scalers(df):
    """
    Fit min-max scaling parameters using training years only.
    """
    train_df = df[df["YEAR"].isin(TRAIN_YEARS)]

    if train_df.empty:
        raise ValueError("No training-year observations were found.")

    scalers = {}

    all_features = list(
        dict.fromkeys(
            CONDITION_FEATURES + [TARGET_FEATURE]
        )
    )

    for feature in all_features:
        if feature in train_df.columns:
            col_min = float(train_df[feature].min())
            col_max = float(train_df[feature].max())

            if col_max == col_min:
                col_max = col_min + 1.0

            scalers[feature] = {
                "min": col_min,
                "max": col_max,
            }

    print(
        f"Scaler fitted on {len(train_df):,} training rows."
    )
    print(
        f"Features fitted: {len(scalers)}"
    )

    return scalers


scalers = compute_scalers(df)

scaler_path = os.path.join(
    OUTPUT_DIR,
    "scaler_params.json"
)

with open(
    scaler_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        scalers,
        f,
        indent=2
    )

print(
    f"Scaler saved to: {scaler_path}"
)


Scaler fitted on 88,394 training rows.
Features fitted: 10
Scaler saved to: Dataset/scaler_params.json


## 7. Define scaling and padding utilities

Two helper functions are used:

- `scale_col()` applies the fitted min-max transformation.
- `pad_or_truncate()` converts variable-length daily arrays into fixed-length arrays.


In [ ]:
def scale_col(series, scalers, feature):
    """
    Apply the fitted min-max transformation to one feature.
    """
    s_min = scalers[feature]["min"]
    s_max = scalers[feature]["max"]
    return (series - s_min) / (s_max - s_min)


def pad_or_truncate(arr, target_len):
    """
    Pad or truncate a 1-D or 2-D array to target_len.
    """
    if arr.ndim == 1:

        if len(arr) >= target_len:
            return arr[:target_len]

        pad = np.zeros(
            target_len - len(arr),
            dtype=arr.dtype
        )

        return np.concatenate(
            [arr, pad]
        )

    if arr.shape[0] >= target_len:
        return arr[:target_len]

    pad = np.zeros(
        (target_len - arr.shape[0], arr.shape[1]),
        dtype=arr.dtype
    )

    return np.concatenate(
        [arr, pad],
        axis=0
    )


## 8. Build one daily sequence per site

This is the core sequence-construction step.

For each calendar day:

1. Scale the conditioning features.
2. Sort the observations by hour.
3. Skip days with fewer than six daytime observations.
4. Build the conditioning array `X`.
5. Build the target GHI array `Y`.
6. Determine the dominant sky regime.
7. Store the site ID and date.

Expected dimensions:

```text
X       → [days, 16, 10]
Y       → [days, 16]
regime  → [days]
site    → [days]
```


In [ ]:
def build_sequences_for_site(site_df, site_name, scalers):
    """
    Build one fixed-length sequence per valid day for one site.
    """
    site_df = site_df.copy()

    # Scale conditioning features.
    for feature in CONDITION_FEATURES:
        if (
            feature in site_df.columns
            and feature in scalers
        ):
            site_df[f"{feature}_SC"] = scale_col(
                site_df[feature],
                scalers,
                feature
            )

    # Scale target GHI.
    target_scaled = f"{TARGET_FEATURE}_SC"

    if target_scaled not in site_df.columns:
        site_df[target_scaled] = scale_col(
            site_df[TARGET_FEATURE],
            scalers,
            TARGET_FEATURE
        )

    condition_columns = [
        f"{feature}_SC"
        for feature in CONDITION_FEATURES
        if f"{feature}_SC" in site_df.columns
    ]

    site_id = SITE_MAP.get(
        site_name,
        -1
    )

    # Group observations by calendar date.
    site_df["DATE"] = pd.to_datetime(
        site_df["DATETIME"]
    ).dt.date

    grouped = site_df.groupby("DATE")

    X_list = []
    Y_list = []
    R_list = []
    S_list = []
    DT_list = []

    skipped = 0

    for date, day_df in grouped:

        day_df = day_df.sort_values("HOUR")

        # Require at least six daytime observations.
        if len(day_df) < 6:
            skipped += 1
            continue

        # Conditioning array.
        cond_arr = (
            day_df[condition_columns]
            .values
            .astype(np.float32)
        )

        cond_arr = pad_or_truncate(
            cond_arr,
            LOOKBACK
        )

        # Target GHI trajectory.
        target_arr = (
            day_df[target_scaled]
            .values
            .astype(np.float32)
        )

        target_arr = pad_or_truncate(
            target_arr,
            HORIZON
        )

        # Dominant sky regime for the day.
        regime_values = (
            day_df["SKY_REGIME"]
            .map(REGIME_MAP)
            .dropna()
            .values
            .astype(int)
        )

        if len(regime_values) == 0:
            skipped += 1
            continue

        regime_mode = int(
            np.bincount(
                regime_values
            ).argmax()
        )

        X_list.append(cond_arr)
        Y_list.append(target_arr)
        R_list.append(regime_mode)
        S_list.append(site_id)
        DT_list.append(str(date))

    if not X_list:
        print(
            f"WARNING: No sequences for {site_name}"
        )
        return None

    result = {
        "X": np.stack(
            X_list,
            axis=0
        ),
        "Y": np.stack(
            Y_list,
            axis=0
        ),
        "regime": np.array(
            R_list,
            dtype=np.int32
        ),
        "site": np.array(
            S_list,
            dtype=np.int32
        ),
        "datetimes": DT_list,
    }

    regime_names = {
        value: key
        for key, value in REGIME_MAP.items()
    }

    unique, counts = np.unique(
        result["regime"],
        return_counts=True
    )

    regime_str = ", ".join(
        regime_names.get(
            int(u),
            str(u)
        )
        + ": "
        + str(int(c))
        for u, c in zip(
            unique,
            counts
        )
    )

    print(
        f"{site_name:<20}"
        f" days: {len(X_list):,}"
        f"  skipped: {skipped:,}"
        f"  X: {result['X'].shape}"
        f"  regimes: [{regime_str}]"
    )

    return result


## 9. Build sequences for all study sites


In [ ]:
print("=== Building daily sequences per site ===")

all_data = {}

for site_name in SITE_MAP.keys():

    site_df = df[
        df["SITE_NAME"] == site_name
    ].copy()

    if site_df.empty:
        print(
            f"WARNING: No data for {site_name}"
        )
        continue

    all_data[site_name] = (
        build_sequences_for_site(
            site_df,
            site_name,
            scalers
        )
    )

print("")
print(
    f"Sites processed: "
    f"{len(all_data)} / {len(SITE_MAP)}"
)


=== Building daily sequences per site ===
Phoenix_AZ           days: 1,826  skipped: 0  X: (1826, 16, 10)  regimes: [Clear: 1791, Cloudy: 31, Overcast: 4]
LosAngeles_CA        days: 1,826  skipped: 0  X: (1826, 16, 10)  regimes: [Clear: 1757, Cloudy: 65, Overcast: 4]
Denver_CO            days: 1,826  skipped: 0  X: (1826, 16, 10)  regimes: [Clear: 1733, Cloudy: 86, Overcast: 7]
Miami_FL             days: 1,826  skipped: 0  X: (1826, 16, 10)  regimes: [Clear: 1770, Cloudy: 53, Overcast: 3]
Seattle_WA           days: 1,826  skipped: 0  X: (1826, 16, 10)  regimes: [Clear: 1442, Cloudy: 377, Overcast: 7]

Sites processed: 5 / 5


## 10. Split into training, validation, and test sets

The original temporal split is preserved:

| Split | Period |
|---|---|
| Train | 2018–2021 |
| Validation | 2022 January–June |
| Test | 2022 July–December |

The sequences are assigned according to their calendar date.


In [ ]:
def split_and_save(all_data):
    """
    Split daily sequences and save train/validation/test NPZ files.
    """
    splits = {
        "train": ([], [], [], []),
        "val": ([], [], [], []),
        "test": ([], [], [], []),
    }

    for site_name, data in all_data.items():

        if data is None:
            continue

        datetimes = pd.to_datetime(
            data["datetimes"]
        )

        years = datetimes.year.values
        months = datetimes.month.values

        masks = {
            "train": np.isin(
                years,
                TRAIN_YEARS
            ),
            "val": (
                (years == 2022)
                & (months <= 6)
            ),
            "test": (
                (years == 2022)
                & (months >= 7)
            ),
        }

        for split_name, mask in masks.items():

            if mask.sum() == 0:
                continue

            X_list, Y_list, R_list, S_list = (
                splits[split_name]
            )

            X_list.append(
                data["X"][mask]
            )
            Y_list.append(
                data["Y"][mask]
            )
            R_list.append(
                data["regime"][mask]
            )
            S_list.append(
                data["site"][mask]
            )

    counts = {}

    print("")
    print(
        f"Saving to {OUTPUT_DIR}/..."
    )
    print("")

    for split_name, (
        X_list,
        Y_list,
        R_list,
        S_list
    ) in splits.items():

        if not X_list:
            print(
                f"  {split_name}: empty, skipping."
            )
            counts[split_name] = 0
            continue

        X = np.concatenate(
            X_list,
            axis=0
        )
        Y = np.concatenate(
            Y_list,
            axis=0
        )
        R = np.concatenate(
            R_list,
            axis=0
        )
        S = np.concatenate(
            S_list,
            axis=0
        )

        output_path = os.path.join(
            OUTPUT_DIR,
            f"sequences_{split_name}.npz"
        )

        np.savez_compressed(
            output_path,
            X=X,
            Y=Y,
            regime=R,
            site=S
        )

        regime_names = {
            value: key
            for key, value in REGIME_MAP.items()
        }

        unique, regime_counts = np.unique(
            R,
            return_counts=True
        )

        regime_str = ", ".join(
            regime_names.get(
                int(u),
                str(u)
            )
            + ": "
            + str(int(c))
            for u, c in zip(
                unique,
                regime_counts
            )
        )

        print(
            f"{split_name:<8}"
            f" -> sequences_{split_name}.npz"
        )

        print(
            f"    X shape : {X.shape}"
            " (days x hours x features)"
        )

        print(
            f"    Y shape : {Y.shape}"
            " (days x horizon hours)"
        )

        print(
            f"    Regimes : {regime_str}"
        )
        print("")

        counts[split_name] = len(X)

    return counts


counts = split_and_save(all_data)



Saving to Dataset/...

train    -> sequences_train.npz
    X shape : (7305, 16, 10) (days x hours x features)
    Y shape : (7305, 16) (days x horizon hours)
    Regimes : Clear: 6782, Cloudy: 501, Overcast: 22

val      -> sequences_val.npz
    X shape : (905, 16, 10) (days x hours x features)
    Y shape : (905, 16) (days x horizon hours)
    Regimes : Clear: 839, Cloudy: 65, Overcast: 1

test     -> sequences_test.npz
    X shape : (920, 16, 10) (days x hours x features)
    Y shape : (920, 16) (days x horizon hours)
    Regimes : Clear: 872, Cloudy: 46, Overcast: 2



## 11. Save the sequence-generation summary

This JSON records the sequence configuration, encodings, temporal split, and final number of sequences.


In [ ]:
summary = {
    "lookback_hours": LOOKBACK,
    "horizon_hours": HORIZON,
    "n_condition_features": len([
        feature
        for feature in CONDITION_FEATURES
        if feature in df.columns
    ]),
    "condition_features": CONDITION_FEATURES,
    "target_feature": TARGET_FEATURE,
    "regime_map": REGIME_MAP,
    "site_map": SITE_MAP,
    "train_years": TRAIN_YEARS,
    "validation_period": "2022 Jan-Jun",
    "test_period": "2022 Jul-Dec",
    "n_train": counts.get("train", 0),
    "n_val": counts.get("val", 0),
    "n_test": counts.get("test", 0),
    "total_sequences": sum(counts.values()),
}

summary_path = os.path.join(
    OUTPUT_DIR,
    "sequence_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    f"Summary saved to: {summary_path}"
)


Summary saved to: Dataset/sequence_summary.json


## 12. Inspect generated sequence shapes

Check the final dimensions before moving to model construction.


In [ ]:
print("=== Generated sequence datasets ===")

for split_name in [
    "train",
    "val",
    "test"
]:

    path = os.path.join(
        OUTPUT_DIR,
        f"sequences_{split_name}.npz"
    )

    if not os.path.exists(path):
        print(
            f"{split_name}: file not found"
        )
        continue

    data = np.load(path)

    print(f"\n{split_name.upper()}")
    print(
        "  X      :",
        data["X"].shape
    )
    print(
        "  Y      :",
        data["Y"].shape
    )
    print(
        "  regime :",
        data["regime"].shape
    )
    print(
        "  site   :",
        data["site"].shape
    )


=== Generated sequence datasets ===

TRAIN
  X      : (7305, 16, 10)
  Y      : (7305, 16)
  regime : (7305,)
  site   : (7305,)

VAL
  X      : (905, 16, 10)
  Y      : (905, 16)
  regime : (905,)
  site   : (905,)

TEST
  X      : (920, 16, 10)
  Y      : (920, 16)
  regime : (920,)
  site   : (920,)


## 13. Final output verification

Expected files:

- `sequences_train.npz`
- `sequences_val.npz`
- `sequences_test.npz`
- `scaler_params.json`
- `sequence_summary.json`

Once these files are present and the array dimensions look correct, Step 3 is complete.

### Next step

Proceed to **Step 4 — Build the GenerativeAURORA diffusion model architecture**.


In [ ]:
expected_outputs = [
    "sequences_train.npz",
    "sequences_val.npz",
    "sequences_test.npz",
    "scaler_params.json",
    "sequence_summary.json",
]

print("=== Step 3 output check ===")

for filename in expected_outputs:

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(path):

        size_kb = round(
            os.path.getsize(path) / 1024,
            1
        )

        print(
            f"[OK]      "
            f"{filename.ljust(30)} "
            f"{size_kb} KB"
        )

    else:
        print(
            f"[MISSING] "
            f"{filename}"
        )

print("")
print("=== Step 3 complete ===")
print(
    f"Train : {counts.get('train', 0):,} day-sequences"
)
print(
    f"Val   : {counts.get('val', 0):,} day-sequences"
)
print(
    f"Test  : {counts.get('test', 0):,} day-sequences"
)
print(
    f"Total : {sum(counts.values()):,} day-sequences"
)
print("")
print("Next: step4_build_model")


=== Step 3 output check ===
[OK]      sequences_train.npz            2360.5 KB
[OK]      sequences_val.npz              304.9 KB
[OK]      sequences_test.npz             293.5 KB
[OK]      scaler_params.json             0.6 KB
[OK]      sequence_summary.json          0.7 KB

=== Step 3 complete ===
Train : 7,305 day-sequences
Val   : 905 day-sequences
Test  : 920 day-sequences
Total : 9,130 day-sequences

Next: step4_build_model
